# nf_sweep_v2 Direct EMA vs Post-hoc EMA

This notebook checks whether the bad EMA curve is because EMA itself is bad, or because post-hoc EMA reconstruction is bad.

It compares one run using the same `train_full` sampler:

- `raw`: checkpoint weights, no EMA.
- `direct_ema0p02`: the directly saved EMA snapshot for the trained profile `ema_sigma_rels[0] = 0.02` from the latest checkpoint.
- `direct_ema0p10`: the directly saved EMA snapshot for `ema_sigma_rels[1] = 0.10` from the latest checkpoint.
- `posthoc_ema0p02`: post-hoc reconstructed EMA target 0.02 pooled across checkpoint EMA snapshots.
- `posthoc_ema0p10`: post-hoc reconstructed EMA target 0.10 pooled across checkpoint EMA snapshots.

Important: the cosmodiff generation is run in a subprocess using `/home/jiamingp/venvs/cosmodiff_nf/bin/python`, so the notebook kernel does not need to import the correct `diffusers` version.

In [ ]:
from pathlib import Path
import os
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path('/home/jiamingp/diffusion_models_repo')
COSMODIFF_DIR = Path('/home/jiamingp/Diffusion_model/cosmo_diffusion_main')
PYTHON_BIN = Path('/home/jiamingp/venvs/cosmodiff_nf/bin/python')

RUN_NAME = 'nf_sweep_v2_u128_n500_e100_nick_default'
CHECKPOINT_ROOT = Path('/scratch/huterer_root/huterer0/jiamingp/saved_runs/nf_sweep_v2') / f'{RUN_NAME}_checkpoints'
CONFIG_PATH = PROJECT_DIR / 'local/nf_sweep_v2/configs' / f'{RUN_NAME}.yaml'
OUT_DIR = PROJECT_DIR / 'results/nf_sweep_v2/ema_direct_vs_posthoc'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 123
N_SAMPLES = 32
BATCH_SIZE = 8
DEVICE = 'cuda'
OVERWRITE = False

print('project:', PROJECT_DIR)
print('cosmodiff:', COSMODIFF_DIR)
print('python:', PYTHON_BIN)
print('run:', RUN_NAME)
print('checkpoint root:', CHECKPOINT_ROOT)
print('config:', CONFIG_PATH)
print('output:', OUT_DIR)

In [ ]:
def cosmodiff_subprocess_env() -> dict[str, str]:
    env = os.environ.copy()

    # Same sklearn stub idea used by the Slurm scripts. This avoids optional
    # transformer/sklearn imports from breaking on Great Lakes libstdc++.
    stub_root = PROJECT_DIR / 'results/cache/python_stubs'
    (stub_root / 'sklearn/metrics').mkdir(parents=True, exist_ok=True)
    (stub_root / 'sklearn/__init__.py').write_text('from . import metrics\n')
    (stub_root / 'sklearn/metrics/__init__.py').write_text(
        "def roc_curve(*args, **kwargs):\n"
        "    raise RuntimeError('sklearn.metrics.roc_curve is stubbed for cosmodiff sampling')\n"
    )

    env['COSMODIFF_DIR'] = str(COSMODIFF_DIR)
    env['COSMODIFF_STUB_SKLEARN'] = '1'
    env['TORCHDYNAMO_DISABLE'] = '1'
    env['PYTHONPATH'] = ':'.join([
        str(stub_root),
        str(COSMODIFF_DIR),
        str(PROJECT_DIR),
        env.get('PYTHONPATH', ''),
    ])
    return env

check_code = """
import inspect
import diffusers
from diffusers import AutoModel
from cosmodiff import optim
from cosmodiff.optim import generate, synthesize_ema_from_checkpoints, load_ema_snapshot
from cosmodiff.utils import find_latest_checkpoint, load_checkpoint
print('OK subprocess environment')
print('diffusers', diffusers.__version__, diffusers.__file__)
print('optim', inspect.getsourcefile(optim))
print('latest checkpoint helper', find_latest_checkpoint)
"""

subprocess.run(
    [str(PYTHON_BIN), '-c', check_code],
    cwd=PROJECT_DIR,
    env=cosmodiff_subprocess_env(),
    check=True,
)

In [ ]:
GEN_SCRIPT = OUT_DIR / '_generate_direct_vs_posthoc.py'
GEN_SCRIPT.write_text("\nfrom __future__ import annotations\n\nimport copy\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport yaml\n\nfrom cosmodiff.utils import find_latest_checkpoint, load_checkpoint\nfrom cosmodiff.optim import generate, synthesize_ema_from_checkpoints, load_ema_snapshot\n\nPROJECT_DIR = Path('/home/jiamingp/diffusion_models_repo')\nRUN_NAME = 'nf_sweep_v2_u128_n500_e100_nick_default'\nCHECKPOINT_ROOT = Path('/scratch/huterer_root/huterer0/jiamingp/saved_runs/nf_sweep_v2') / f'{RUN_NAME}_checkpoints'\nCONFIG_PATH = PROJECT_DIR / 'local/nf_sweep_v2/configs' / f'{RUN_NAME}.yaml'\nOUT_DIR = PROJECT_DIR / 'results/nf_sweep_v2/ema_direct_vs_posthoc'\nOUT_DIR.mkdir(parents=True, exist_ok=True)\n\nSEED = 123\nN_SAMPLES = 32\nBATCH_SIZE = 8\nDEVICE = torch.device('cuda')\nOVERWRITE = False\n\nwith open(CONFIG_PATH) as f:\n    cfg = yaml.safe_load(f)\n\nlatest_ckpt_raw = find_latest_checkpoint(str(CHECKPOINT_ROOT))\nif latest_ckpt_raw is None:\n    raise FileNotFoundError(f'No checkpoints found under {CHECKPOINT_ROOT}')\nlatest_ckpt = Path(latest_ckpt_raw)\n\nsample_size = cfg['model']['kwargs']['sample_size']\nif isinstance(sample_size, list):\n    image_shape = (cfg['model']['kwargs'].get('in_channels', 1), *sample_size)\nelse:\n    image_shape = (cfg['model']['kwargs'].get('in_channels', 1), sample_size, sample_size)\n\nprint('latest checkpoint:', latest_ckpt)\nprint('image_shape:', image_shape)\n\n\ndef unwrap_ema_model(model):\n    # ema-pytorch may return a KarrasEMA wrapper from synthesize_ema_model.\n    return model.ema_model if hasattr(model, 'ema_model') else model\n\n\ndef load_raw_model():\n    model, scheduler, *_ = load_checkpoint(str(latest_ckpt))\n    return model, scheduler\n\n\n@torch.no_grad()\ndef sample_model(label: str, model, scheduler) -> None:\n    out_path = OUT_DIR / f'{RUN_NAME}_{label}_train_full.npz'\n    if out_path.exists() and not OVERWRITE:\n        print('exists:', out_path)\n        return\n\n    model = unwrap_ema_model(model).to(DEVICE).eval()\n    all_samples = []\n    remaining = N_SAMPLES\n    offset = 0\n    batch_index = 0\n\n    while remaining > 0:\n        bs = min(BATCH_SIZE, remaining)\n        # Same seed schedule for each label, so raw/direct/posthoc use matching noise.\n        generator = torch.Generator(device=DEVICE).manual_seed(SEED + batch_index)\n        samples = generate(\n            model=model,\n            noise_scheduler=copy.deepcopy(scheduler),\n            batch_size=bs,\n            image_shape=image_shape,\n            device=DEVICE,\n            generator=generator,\n        )\n        all_samples.append(samples.detach().cpu().numpy())\n        offset += bs\n        remaining -= bs\n        batch_index += 1\n        print(f'{label}: {offset}/{N_SAMPLES}', flush=True)\n\n    arr = np.concatenate(all_samples, axis=0)\n    np.savez(\n        out_path,\n        samples=arr,\n        label=np.array(label),\n        ckpt_path=np.array(str(latest_ckpt)),\n        checkpoint_root=np.array(str(CHECKPOINT_ROOT)),\n    )\n    print('wrote', out_path, arr.shape, flush=True)\n\n\n# 1. Raw checkpoint weights.\nmodel, scheduler = load_raw_model()\nsample_model('raw', model, scheduler)\n\n# 2. Direct saved EMA snapshots from the latest checkpoint.\n# profile_index 0 -> ema_sigma_rels[0] = 0.02\n# profile_index 1 -> ema_sigma_rels[1] = 0.10\nfor label, profile_index in [('direct_ema0p02', 0), ('direct_ema0p10', 1)]:\n    model, scheduler = load_raw_model()\n    model = load_ema_snapshot(model, str(latest_ckpt), profile_index=profile_index)\n    sample_model(label, model, scheduler)\n\n# 3. Post-hoc reconstructed EMA from all checkpoint EMA snapshots.\nfor label, sigma_rel in [('posthoc_ema0p02', 0.02), ('posthoc_ema0p10', 0.10)]:\n    model, scheduler = load_raw_model()\n    model = synthesize_ema_from_checkpoints(\n        model,\n        str(CHECKPOINT_ROOT),\n        sigma_rel_target=sigma_rel,\n    )\n    sample_model(label, model, scheduler)\n")
print('wrote generation script:', GEN_SCRIPT)

In [ ]:
# Run generation in the correct cosmodiff Python environment.
# This writes .npz files under OUT_DIR and skips files that already exist.
subprocess.run(
    [str(PYTHON_BIN), str(GEN_SCRIPT)],
    cwd=PROJECT_DIR,
    env=cosmodiff_subprocess_env(),
    check=True,
)

In [ ]:
# Load generated arrays back into this notebook kernel.
LABELS = ['raw', 'direct_ema0p02', 'direct_ema0p10', 'posthoc_ema0p02', 'posthoc_ema0p10']

sample_sets = {}
for label in LABELS:
    sample_path = OUT_DIR / f'{RUN_NAME}_{label}_train_full.npz'
    if not sample_path.exists():
        raise FileNotFoundError(sample_path)
    arr = np.load(sample_path)['samples']
    sample_sets[label] = arr
    print(label, arr.shape, arr.mean(), arr.std())

In [ ]:
# Load real data and compute metrics. This uses the local fallback loader if the notebook kernel cannot import cosmodiff.
from simdiff_eval.io import load_real_from_config
from simdiff_eval.metrics import batch_power_spectra, field_histogram, power_spectrum_summary

MAX_REAL_RAW_CUBES = 16  # 16 raw cubes -> 512 2D slices with zthin=4
PK_NBINS = 25
HIST_BINS = 120

real = load_real_from_config(CONFIG_PATH, max_raw_samples=MAX_REAL_RAW_CUBES)
print('real', real.shape, real.mean(), real.std())


def onepoint_l1(real_arr: np.ndarray, gen_arr: np.ndarray, bins: int = HIST_BINS) -> float:
    real_h = field_histogram(real_arr, bins=bins)
    gen_h = field_histogram(gen_arr, bins=bins)
    edges = np.asarray(real_h['bin_edges'])
    dx = np.diff(edges)
    return float(np.sum(np.abs(np.asarray(real_h['hist']) - np.asarray(gen_h['hist'])) * dx))


rows = []
pk_cache = {}
for label, arr in sample_sets.items():
    pk_summary = power_spectrum_summary(real, arr, nbins=PK_NBINS)
    pk_real, kbins = batch_power_spectra(real, nbins=PK_NBINS)
    pk_gen, _ = batch_power_spectra(arr, nbins=PK_NBINS)
    real_mean = np.nanmean(pk_real, axis=0)
    gen_mean = np.nanmean(pk_gen, axis=0)
    pk_cache[label] = (kbins, real_mean, gen_mean)
    rows.append({
        'label': label,
        'n_generated': len(arr),
        'onepoint_l1': onepoint_l1(real, arr),
        **pk_summary,
        'generated_mean': float(arr.mean()),
        'generated_std': float(arr.std()),
        'std_ratio': float(arr.std() / np.clip(real.std(), 1e-30, None)),
    })

metric_df = pd.DataFrame(rows).sort_values('pk_log10_mae')
display(metric_df)
metric_path = OUT_DIR / f'{RUN_NAME}_direct_vs_posthoc_metrics.csv'
metric_df.to_csv(metric_path, index=False)
print('wrote', metric_path)

In [ ]:
# Plot one-point histogram and P(k) percent difference.
real_hist = field_histogram(real, bins=HIST_BINS)
edges = np.asarray(real_hist['bin_edges'])
centers = 0.5 * (edges[:-1] + edges[1:])

fig, axes = plt.subplots(2, len(LABELS), figsize=(4.4 * len(LABELS), 7.2), squeeze=False)
for col, label in enumerate(LABELS):
    arr = sample_sets[label]
    gen_hist = field_histogram(arr, bins=HIST_BINS)

    ax = axes[0, col]
    ax.plot(centers, real_hist['hist'], color='black', linewidth=2, label='real')
    ax.plot(centers, gen_hist['hist'], color='tab:blue', linewidth=1.8, label=label)
    ax.set_yscale('log')
    ax.set_title(label)
    ax.set_xlabel('normalized field value')
    ax.set_ylabel('density')
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)

    kbins, pk_real_mean, pk_gen_mean = pk_cache[label]
    ratio_pct = 100.0 * (pk_gen_mean - pk_real_mean) / np.clip(pk_real_mean, 1e-30, None)

    ax = axes[1, col]
    ax.plot(kbins, ratio_pct, marker='o', color='tab:blue')
    ax.axhline(0.0, color='black', linestyle=':', linewidth=1.2)
    ax.set_xlabel('k bin')
    ax.set_ylabel('100 * (P_gen - P_real) / P_real [%]')
    ax.grid(alpha=0.25)

fig.suptitle(f'{RUN_NAME}: direct saved EMA vs post-hoc EMA, train_full')
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig_path = OUT_DIR / f'{RUN_NAME}_direct_vs_posthoc_ema.png'
fig.savefig(fig_path, dpi=180)
print('wrote', fig_path)

## How to interpret

- If `direct_ema0p02` and `posthoc_ema0p02` are both bad, then EMA itself is hurting this setup.
- If `direct_ema0p02` is good but `posthoc_ema0p02` is bad, then post-hoc reconstruction is the problem.
- If direct and post-hoc are close to each other and both worse than `raw`, then the raw final checkpoint really is better for this run/metric.
- If direct EMA is much better than raw but the original EMA sweep plot was bad, then the sampling path for the old EMA files was wrong.